In [1]:
from pathlib import Path
from collections import Counter
from datetime import datetime
import json
import os
import re
import shutil
import subprocess
import time
import zipfile

import pandas as pd
import requests
import sacrebleu
from IPython.display import display
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / 'results').exists(), f'项目根目录识别失败: {PROJECT_ROOT}'

BASELINE_DIR = PROJECT_ROOT / 'results' / 'fourlang_base_model_selection' / 'beam5_accuracy'
RESULT_DIR = PROJECT_ROOT / 'results' / 'qwen3_4b_q8_uzbek_selection'
MODEL_DIR = PROJECT_ROOT / 'models' / 'huggingface' / 'qwen3-4b-gguf'
LLAMA_ROOT = PROJECT_ROOT / 'tools' / 'llama.cpp'
LLAMA_DOWNLOAD_DIR = LLAMA_ROOT / 'downloads'
LLAMA_RUNTIME_DIR = LLAMA_ROOT / 'runtime_cuda_13_3'
for path in [RESULT_DIR, MODEL_DIR, LLAMA_DOWNLOAD_DIR, LLAMA_RUNTIME_DIR]:
    path.mkdir(parents=True, exist_ok=True)

MODEL_REPO = 'Qwen/Qwen3-4B-GGUF'
MODEL_FILENAME = 'Qwen3-4B-Q8_0.gguf'
MODEL_FILE = MODEL_DIR / MODEL_FILENAME
LLAMA_CUDA_VERSION = '13.3'
SERVER_HOST = '127.0.0.1'
SERVER_PORT = 18081
SERVER_URL = f'http://{SERVER_HOST}:{SERVER_PORT}'
SAVE_EVERY = 10
REQUEST_TIMEOUT_SECONDS = 180
MODEL_ALIAS = 'qwen3-4b-q8'

print('项目目录:', PROJECT_ROOT)
print('模型将保存到:', MODEL_FILE)
print('结果将保存到:', RESULT_DIR)


项目目录: D:\dev\projects\fourlang_translation
模型将保存到: D:\dev\projects\fourlang_translation\models\huggingface\qwen3-4b-gguf\Qwen3-4B-Q8_0.gguf
结果将保存到: D:\dev\projects\fourlang_translation\results\qwen3_4b_q8_uzbek_selection


D:\dev\projects\fourlang_translation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def download_with_resume(url, destination, expected_size=None):
    destination = Path(destination)
    partial = destination.with_suffix(destination.suffix + '.part')
    if destination.exists() and (expected_size is None or destination.stat().st_size == expected_size):
        print('已存在，跳过下载:', destination.name)
        return destination
    existing = partial.stat().st_size if partial.exists() else 0
    headers = {'User-Agent': 'fourlang-translation-eval'}
    if existing:
        headers['Range'] = f'bytes={existing}-'
    with requests.get(url, headers=headers, stream=True, timeout=(30, 300)) as response:
        response.raise_for_status()
        append = existing > 0 and response.status_code == 206
        if not append:
            existing = 0
        total = expected_size or (existing + int(response.headers.get('content-length', 0)))
        mode = 'ab' if append else 'wb'
        with open(partial, mode) as file, tqdm(total=total, initial=existing, unit='B', unit_scale=True, desc=destination.name) as bar:
            for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                if chunk:
                    file.write(chunk)
                    bar.update(len(chunk))
    if expected_size and partial.stat().st_size != expected_size:
        raise RuntimeError(f'下载大小不匹配: {partial.stat().st_size} != {expected_size}')
    partial.replace(destination)
    return destination

def find_llama_server():
    matches = list(LLAMA_RUNTIME_DIR.rglob('llama-server.exe'))
    return matches[0] if matches else None

LLAMA_SERVER = find_llama_server()
if LLAMA_SERVER is None:
    latest_url = 'https://github.com/ggml-org/llama.cpp/releases/latest'
    try:
        latest_response = requests.get(latest_url, headers={'User-Agent': 'fourlang-translation-eval'}, timeout=60)
        latest_response.raise_for_status()
        release_tag = latest_response.url.rstrip('/').split('/')[-1]
        if not re.fullmatch(r'b\d+', release_tag):
            raise ValueError(f'无法从重定向地址识别版本: {latest_response.url}')
    except Exception as error:
        release_tag = 'b10516'
        print(f'自动识别最新版本失败，改用已验证版本 {release_tag}: {error}')
    release_url = f'https://github.com/ggml-org/llama.cpp/releases/tag/{release_tag}'
    asset_base_url = f'https://github.com/ggml-org/llama.cpp/releases/download/{release_tag}'
    main_name = f'llama-{release_tag}-bin-win-cuda-{LLAMA_CUDA_VERSION}-x64.zip'
    cudart_name = f'cudart-llama-bin-win-cuda-{LLAMA_CUDA_VERSION}-x64.zip'
    main_asset = {'name': main_name, 'browser_download_url': f'{asset_base_url}/{main_name}', 'size': None}
    cudart_asset = {'name': cudart_name, 'browser_download_url': f'{asset_base_url}/{cudart_name}', 'size': None}
    print('选用 llama.cpp:', release_tag)
    archives = []
    for asset in [main_asset, cudart_asset]:
        archive = download_with_resume(asset['browser_download_url'], LLAMA_DOWNLOAD_DIR / asset['name'], asset.get('size'))
        archives.append(archive)
    for archive in archives:
        print('解压:', archive.name)
        with zipfile.ZipFile(archive) as package:
            package.extractall(LLAMA_RUNTIME_DIR)
    LLAMA_SERVER = find_llama_server()
    runtime_manifest = {
        'release_tag': release_tag,
        'release_url': release_url,
        'cuda_version': LLAMA_CUDA_VERSION,
        'assets': [a['name'] for a in [main_asset, cudart_asset]],
        'downloaded_at': datetime.now().isoformat(timespec='seconds'),
    }
    (LLAMA_ROOT / 'runtime_manifest.json').write_text(json.dumps(runtime_manifest, ensure_ascii=False, indent=2), encoding='utf-8')
assert LLAMA_SERVER and LLAMA_SERVER.exists(), 'llama-server.exe 准备失败'
print('llama-server:', LLAMA_SERVER)


自动识别最新版本失败，改用已验证版本 b10516: 无法从重定向地址识别版本: https://github.com/ggml-org/llama.cpp/releases/tag/v0.2.0
选用 llama.cpp: b10516


llama-b10516-bin-win-cuda-13.3-x64.zip: 100%|██████████| 147M/147M [00:14<00:00, 10.4MB/s] 
cudart-llama-bin-win-cuda-13.3-x64.zip: 100%|██████████| 391M/391M [01:00<00:00, 6.50MB/s] 


解压: llama-b10516-bin-win-cuda-13.3-x64.zip
解压: cudart-llama-bin-win-cuda-13.3-x64.zip
llama-server: D:\dev\projects\fourlang_translation\tools\llama.cpp\runtime_cuda_13_3\llama-server.exe


In [3]:
from huggingface_hub import hf_hub_download

if not MODEL_FILE.exists() or MODEL_FILE.stat().st_size < 4_000_000_000:
    print('开始下载 Qwen3-4B Q8，约 4.28 GB；中断后重新运行会续传。')
    downloaded_model = hf_hub_download(
        repo_id=MODEL_REPO,
        filename=MODEL_FILENAME,
        local_dir=MODEL_DIR,
    )
    MODEL_FILE = Path(downloaded_model).resolve()
assert MODEL_FILE.exists(), f'模型文件不存在: {MODEL_FILE}'
assert MODEL_FILE.stat().st_size > 4_000_000_000, f'模型文件可能不完整: {MODEL_FILE.stat().st_size:,} bytes'
print(f'模型准备完成: {MODEL_FILE} ({MODEL_FILE.stat().st_size / 1024**3:.2f} GiB)')


开始下载 Qwen3-4B Q8，约 4.28 GB；中断后重新运行会续传。


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


模型准备完成: D:\dev\projects\fourlang_translation\models\huggingface\qwen3-4b-gguf\Qwen3-4B-Q8_0.gguf (3.99 GiB)


In [4]:
benchmark_path = BASELINE_DIR / 'flores_fourlang_600.csv'
baseline_paths = {
    'm2m100_418m': BASELINE_DIR / 'predictions_m2m100_418m.csv',
    'm2m100_1_2b': BASELINE_DIR / 'predictions_m2m100_1_2b.csv',
}
assert benchmark_path.exists(), f'缺少基准集: {benchmark_path}'
for name, path in baseline_paths.items():
    assert path.exists(), f'缺少 {name} 预测: {path}'

all_benchmark = pd.read_csv(benchmark_path, keep_default_na=False)
benchmark_df = all_benchmark[(all_benchmark.src_lang == 'uz') | (all_benchmark.tgt_lang == 'uz')].copy()
benchmark_df['direction'] = benchmark_df.src_lang + '-' + benchmark_df.tgt_lang
benchmark_df = benchmark_df.sort_values(['direction', 'eval_id']).reset_index(drop=True)
expected_directions = {'en-uz', 'ru-uz', 'uz-en', 'uz-ru', 'uz-zh', 'zh-uz'}
assert set(benchmark_df.direction) == expected_directions, sorted(benchmark_df.direction.unique())
assert len(benchmark_df) == 300, f'预期 300 条，实际 {len(benchmark_df)} 条'
assert benchmark_df.groupby('direction').size().eq(50).all(), benchmark_df.groupby('direction').size()

baseline_frames = []
wanted_ids = set(benchmark_df.eval_id)
for model_name, path in baseline_paths.items():
    frame = pd.read_csv(path, keep_default_na=False)
    frame = frame[frame.eval_id.isin(wanted_ids)].copy()
    assert len(frame) == 300 and frame.eval_id.nunique() == 300, f'{model_name} 行数不正确: {len(frame)}'
    frame['model_name'] = model_name
    baseline_frames.append(frame)
display(benchmark_df.groupby('direction').size().rename('samples').reset_index())
display(benchmark_df[['eval_id', 'src_lang', 'tgt_lang', 'source', 'reference']].head(3))


,direction,samples
0,en-uz,50
1,ru-uz,50
2,uz-en,50
3,uz-ru,50
4,uz-zh,50
5,zh-uz,50


,eval_id,src_lang,tgt_lang,source,reference
0,en-uz-0063,en,uz,Historians have criticized past FBI policies f...,Tarixchilar agentlikning muvaffaqiyat darajasi...
1,en-uz-0068,en,uz,U.S. President George W. Bush arrived in Singa...,"AQSH Prezidenti Jorj V. Bush, Osiyo bo'ylab bi..."
2,en-uz-0083,en,uz,"The Ninth Ward, which saw flooding as high as ...",Katrina bo'roni vaqtida 20 fut balandlikdagi t...


In [5]:
def server_health():
    try:
        return requests.get(f'{SERVER_URL}/health', timeout=3).status_code == 200
    except requests.RequestException:
        return False

def server_models():
    try:
        response = requests.get(f'{SERVER_URL}/v1/models', timeout=5)
        response.raise_for_status()
        return response.json()
    except requests.RequestException:
        return {}

if server_health():
    model_info = json.dumps(server_models(), ensure_ascii=False)
    if MODEL_ALIAS not in model_info:
        raise RuntimeError(f'端口 {SERVER_PORT} 已被另一个模型服务占用。请先关闭它，再重跑本 Cell。当前模型信息: {model_info}')
    print('复用已经启动的 Qwen3-4B Q8 服务。')
else:
    dll_dirs = sorted({str(path.parent) for path in LLAMA_RUNTIME_DIR.rglob('*.dll')})
    server_env = os.environ.copy()
    server_env['PATH'] = os.pathsep.join(dll_dirs + [str(LLAMA_SERVER.parent), server_env.get('PATH', '')])
    log_path = RESULT_DIR / 'llama_server.log'
    server_log_handle = open(log_path, 'a', encoding='utf-8')
    command = [
        str(LLAMA_SERVER),
        '-m', str(MODEL_FILE),
        '--alias', MODEL_ALIAS,
        '--host', SERVER_HOST,
        '--port', str(SERVER_PORT),
        '-ngl', '99',
        '-c', '2048',
        '--parallel', '1',
        '--jinja',
    ]
    creationflags = subprocess.CREATE_NO_WINDOW if os.name == 'nt' else 0
    server_process = subprocess.Popen(
        command,
        cwd=LLAMA_SERVER.parent,
        env=server_env,
        stdout=server_log_handle,
        stderr=subprocess.STDOUT,
        creationflags=creationflags,
    )
    print('正在加载模型到 GPU，请等待服务就绪……')
    for _ in tqdm(range(180), desc='等待 llama-server'):
        if server_health():
            break
        if server_process.poll() is not None:
            server_log_handle.flush()
            tail = log_path.read_text(encoding='utf-8', errors='replace')[-6000:]
            raise RuntimeError(f'llama-server 提前退出，退出码 {server_process.returncode}。日志末尾:\n{tail}')
        time.sleep(1)
    else:
        raise TimeoutError(f'模型服务 180 秒内未就绪，请查看 {log_path}')
print('服务已就绪:', server_models())


正在加载模型到 GPU，请等待服务就绪……


等待 llama-server:   2%|▏         | 4/180 [00:06<04:28,  1.52s/it]

服务已就绪: {'models': [{'name': 'qwen3-4b-q8', 'model': 'qwen3-4b-q8', 'modified_at': '', 'size': '', 'digest': '', 'type': 'model', 'description': '', 'tags': [''], 'capabilities': ['completion'], 'parameters': '', 'details': {'parent_model': '', 'format': 'gguf', 'family': '', 'families': [''], 'parameter_size': '', 'quantization_level': ''}}], 'object': 'list', 'data': [{'id': 'qwen3-4b-q8', 'aliases': ['qwen3-4b-q8'], 'tags': [], 'object': 'model', 'created': 1787551256, 'owned_by': 'llamacpp', 'meta': {'vocab_type': 2, 'n_vocab': 151936, 'n_ctx': 2048, 'n_ctx_train': 40960, 'n_embd': 2560, 'n_params': 4022468096, 'size': 4274448384, 'ftype': 'Q8_0'}}]}


In [6]:
LANGUAGE_NAMES = {
    'en': 'English',
    'uz': 'Uzbek in Latin script',
    'ru': 'Russian',
    'zh': 'Simplified Chinese',
}
SYSTEM_PROMPT = (
    'You are a professional translation engine. Return only the translated text. '
    'Never explain, answer the source, add notes, or repeat the instructions.'
)

def clean_translation(text):
    text = str(text).strip()
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.S | re.I).strip()
    if '</think>' in text.lower():
        text = re.split(r'</think>', text, flags=re.I)[-1].strip()
    text = re.sub(r'^(translation|translated text|译文|翻译|tarjima|перевод)\s*[:：]\s*', '', text, flags=re.I).strip()
    if len(text) >= 2 and text[0] == text[-1] and text[0] in ['\"', "'", '“', '”']:
        text = text[1:-1].strip()
    return text

def translate_one(source, src_lang, tgt_lang, retries=3):
    user_prompt = (
        '/no_think\n'
        f'Translate the following text from {LANGUAGE_NAMES[src_lang]} to {LANGUAGE_NAMES[tgt_lang]}. '
        'Preserve all names, numbers, dates, units, and the original meaning. '
        'Use natural target-language grammar. Output only the translation.\n\n'
        f'Text:\n{source}'
    )
    payload = {
        'model': MODEL_ALIAS,
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': user_prompt},
        ],
        'temperature': 0,
        'top_p': 1,
        'seed': 42,
        'max_tokens': 256,
        'stream': False,
    }
    last_error = None
    for attempt in range(retries):
        started = time.perf_counter()
        try:
            response = requests.post(f'{SERVER_URL}/v1/chat/completions', json=payload, timeout=REQUEST_TIMEOUT_SECONDS)
            response.raise_for_status()
            body = response.json()
            choice = body['choices'][0]
            raw = choice['message']['content']
            usage = body.get('usage', {})
            return {
                'prediction': clean_translation(raw),
                'raw_prediction': raw,
                'total_seconds': time.perf_counter() - started,
                'generated_tokens': usage.get('completion_tokens', 0),
                'finish_reason': choice.get('finish_reason', ''),
            }
        except Exception as error:
            last_error = error
            if attempt + 1 < retries:
                time.sleep(2 ** attempt)
    raise RuntimeError(f'翻译请求连续失败 {retries} 次: {last_error}')

smoke_row = benchmark_df.iloc[0]
smoke_result = translate_one(smoke_row.source, smoke_row.src_lang, smoke_row.tgt_lang)
print('原文:', smoke_row.source)
print('参考:', smoke_row.reference)
print('Qwen:', smoke_result['prediction'])
print('耗时:', round(smoke_result['total_seconds'], 3), '秒')


原文: Historians have criticized past FBI policies for focusing resources on cases which are easy to solve, especially stolen car cases, with the intent of boosting the agency's success rate.
参考: Tarixchilar agentlikning muvaffaqiyat darajasini kuchaytirish uchun osongina hal qilish mumkin bo'lgan, ayniqsa avtomobil o'g'rilanishi bo'yicha ishlarga resurslarning qaratilganligi uchun FQB agentligining o'tmishdagi siyosatlarini tanqid qildilar.
Qwen: Tarihshunoslar, ҳиссият ҳисоби ҳисобланган ҳисобларга (ҳамма ҳисоблар, ҳамма ҳисоблар, ҳамма ҳисоблар) ҳисобланган ҳисобларга (ҳамма ҳисоблар, ҳамма ҳисоблар, ҳамма ҳисоблар) ҳисобланган ҳисобларга (ҳамма ҳисоблар, ҳамма ҳисоблар, ҳамма ҳисоблар) ҳисобланган ҳисобларга (ҳамма ҳисоблар, ҳамма ҳисоблар, ҳамма ҳисоблар) ҳисобланган ҳисобларга (ҳамма ҳисоблар, ҳамма ҳисоблар, ҳамма ҳисоблар) ҳисоблан
耗时: 3.793 秒


In [7]:
qwen_prediction_path = RESULT_DIR / 'predictions_qwen3_4b_q8.csv'
if qwen_prediction_path.exists():
    saved = pd.read_csv(qwen_prediction_path, keep_default_na=False).to_dict('records')
else:
    saved = []
done_ids = {row['eval_id'] for row in saved}
remaining = benchmark_df[~benchmark_df.eval_id.isin(done_ids)]
print(f'已经完成 {len(done_ids)}/300，本次还需运行 {len(remaining)} 条。')

records = list(saved)
for row in tqdm(remaining.itertuples(index=False), total=len(remaining), desc='Qwen3-4B Q8 翻译'):
    result = translate_one(row.source, row.src_lang, row.tgt_lang)
    record = {
        'eval_id': row.eval_id,
        'benchmark': row.benchmark,
        'pair_id': row.pair_id,
        'src_lang': row.src_lang,
        'tgt_lang': row.tgt_lang,
        'source': row.source,
        'reference': row.reference,
        'model_name': 'qwen3_4b_q8',
        **result,
    }
    records.append(record)
    if len(records) % SAVE_EVERY == 0:
        pd.DataFrame(records).drop_duplicates('eval_id', keep='last').to_csv(qwen_prediction_path, index=False, encoding='utf-8-sig')

qwen_predictions = pd.DataFrame(records).drop_duplicates('eval_id', keep='last')
qwen_predictions = qwen_predictions.sort_values(['src_lang', 'tgt_lang', 'eval_id']).reset_index(drop=True)
qwen_predictions.to_csv(qwen_prediction_path, index=False, encoding='utf-8-sig')
assert len(qwen_predictions) == 300 and qwen_predictions.eval_id.nunique() == 300
print('Qwen 300 条翻译全部完成并保存到:', qwen_prediction_path)


已经完成 0/300，本次还需运行 300 条。


Qwen3-4B Q8 翻译: 100%|██████████| 300/300 [05:18<00:00,  1.06s/it]

Qwen 300 条翻译全部完成并保存到: D:\dev\projects\fourlang_translation\results\qwen3_4b_q8_uzbek_selection\predictions_qwen3_4b_q8.csv


In [8]:
def truthy(series):
    if series.dtype == bool:
        return series
    return series.astype(str).str.lower().isin(['true', '1', 'yes'])

def has_repetition(text):
    tokens = re.findall(r'\w+|[^\w\s]', str(text).lower(), flags=re.UNICODE)
    for size in [3, 4, 5]:
        grams = [tuple(tokens[i:i + size]) for i in range(max(0, len(tokens) - size + 1))]
        if grams and max(Counter(grams).values()) >= 3:
            return True
    return False

def extract_numbers(text):
    values = re.findall(r'(?<!\w)[+-]?\d+(?:[.,:/-]\d+)*(?:%|°)?', str(text))
    return Counter(v.replace(',', '.').replace(' ', '') for v in values)

def output_has_prefix(text):
    return bool(re.match(r'^\s*(translation|translated text|译文|翻译|tarjima|перевод)\s*[:：]', str(text), flags=re.I))

def add_diagnostics(frame):
    frame = frame.copy()
    frame['has_repetition'] = frame.prediction.map(has_repetition)
    frame['hit_max_tokens'] = frame.get('finish_reason', pd.Series('', index=frame.index)).astype(str).eq('length')
    frame['source_has_numbers'] = frame.source.map(lambda value: bool(extract_numbers(value)))
    frame['numbers_preserved'] = [extract_numbers(source) == extract_numbers(prediction) for source, prediction in zip(frame.source, frame.prediction)]
    frame['format_prefix'] = frame.get('raw_prediction', frame.prediction).map(output_has_prefix)
    reference_length = frame.reference.astype(str).str.len().clip(lower=1)
    frame['reference_length_ratio'] = frame.prediction.astype(str).str.len() / reference_length
    frame['abnormal_length'] = (frame.reference_length_ratio < 0.35) | (frame.reference_length_ratio > 2.5)
    return frame

def compute_metrics(frame):
    records = []
    for (model_name, src_lang, tgt_lang), group in frame.groupby(['model_name', 'src_lang', 'tgt_lang']):
        predictions = group.prediction.astype(str).tolist()
        references = group.reference.astype(str).tolist()
        numbered = group[group.source_has_numbers]
        records.append({
            'model_name': model_name,
            'direction': f'{src_lang}-{tgt_lang}',
            'samples': len(group),
            'bleu': sacrebleu.corpus_bleu(predictions, [references], tokenize='zh' if tgt_lang == 'zh' else '13a').score,
            'chrf2': sacrebleu.corpus_chrf(predictions, [references], word_order=2).score,
            'repetition_percent': group.has_repetition.mean() * 100,
            'hit_max_tokens_percent': group.hit_max_tokens.mean() * 100,
            'abnormal_length_percent': group.abnormal_length.mean() * 100,
            'format_prefix_percent': group.format_prefix.mean() * 100,
            'numbered_samples': len(numbered),
            'number_preservation_percent': numbered.numbers_preserved.mean() * 100 if len(numbered) else float('nan'),
            'latency_mean_seconds': pd.to_numeric(group.total_seconds, errors='coerce').mean(),
            'latency_p95_seconds': pd.to_numeric(group.total_seconds, errors='coerce').quantile(.95),
        })
    return pd.DataFrame(records)

all_predictions = pd.concat(baseline_frames + [qwen_predictions], ignore_index=True, sort=False)
all_predictions = add_diagnostics(all_predictions)
metrics = compute_metrics(all_predictions).sort_values(['direction', 'model_name']).reset_index(drop=True)
metrics_path = RESULT_DIR / 'metrics_qwen3_q8_vs_m2m100_uzbek.csv'
metrics.to_csv(metrics_path, index=False, encoding='utf-8-sig')
display(metrics.round(4))


,model_name,direction,samples,bleu,chrf2,repetition_percent,hit_max_tokens_percent,abnormal_length_percent,format_prefix_percent,numbered_samples,number_preservation_percent,latency_mean_seconds,latency_p95_seconds
0,m2m100_1_2b,en-uz,50,0.3830,13.9508,0.0,0.0,2.0,0.0,13,100.0000,0.5167,0.7521
1,m2m100_418m,en-uz,50,0.5164,15.5223,0.0,0.0,0.0,0.0,13,84.6154,0.2999,0.4873
2,qwen3_4b_q8,en-uz,50,2.4699,14.9484,16.0,34.0,52.0,0.0,13,53.8462,2.0493,4.3378
3,m2m100_1_2b,ru-uz,50,0.3447,11.6773,0.0,0.0,14.0,0.0,11,63.6364,0.5811,0.8556
4,m2m100_418m,ru-uz,50,0.3437,15.6118,0.0,0.0,0.0,0.0,11,81.8182,0.3501,0.5148
5,qwen3_4b_q8,ru-uz,50,2.4231,23.1333,14.0,14.0,18.0,0.0,11,72.7273,1.2554,3.6714
6,m2m100_1_2b,uz-en,50,2.2183,20.7388,0.0,0.0,0.0,0.0,13,69.2308,0.7475,1.7707
7,m2m100_418m,uz-en,50,1.6240,22.0329,0.0,0.0,0.0,0.0,13,84.6154,0.4338,0.8238
8,qwen3_4b_q8,uz-en,50,19.0607,43.3740,0.0,0.0,8.0,0.0,13,84.6154,0.4577,0.6819
9,m2m100_1_2b,uz-ru,50,1.2399,18.0590,0.0,0.0,0.0,0.0,13,69.2308,0.7532,1.3837


In [9]:
sentence_rows = []
for row in all_predictions.itertuples(index=False):
    sentence_rows.append({
        'eval_id': row.eval_id,
        'model_name': row.model_name,
        'direction': f'{row.src_lang}-{row.tgt_lang}',
        'sentence_chrf2': sacrebleu.sentence_chrf(str(row.prediction), [str(row.reference)], word_order=2).score,
    })
sentence_scores = pd.DataFrame(sentence_rows)
score_wide = sentence_scores.pivot(index=['eval_id', 'direction'], columns='model_name', values='sentence_chrf2').reset_index()
win_records = []
for direction, group in score_wide.groupby('direction'):
    for baseline_name in ['m2m100_418m', 'm2m100_1_2b']:
        delta = group.qwen3_4b_q8 - group[baseline_name]
        win_records.append({
            'direction': direction,
            'comparison': f'qwen3_4b_q8_minus_{baseline_name}',
            'mean_sentence_chrf2_delta': delta.mean(),
            'qwen_wins': int((delta > 0.5).sum()),
            'ties': int((delta.abs() <= 0.5).sum()),
            'qwen_losses': int((delta < -0.5).sum()),
        })
win_loss = pd.DataFrame(win_records)
win_loss.to_csv(RESULT_DIR / 'sentence_win_loss.csv', index=False, encoding='utf-8-sig')

metric_pivot = metrics.pivot(index='direction', columns='model_name', values='chrf2')
metric_pivot['best_m2m100_chrf2'] = metric_pivot[['m2m100_418m', 'm2m100_1_2b']].max(axis=1)
metric_pivot['qwen_minus_best_m2m100'] = metric_pivot.qwen3_4b_q8 - metric_pivot.best_m2m100_chrf2
direction_comparison = metric_pivot.reset_index()
direction_comparison.to_csv(RESULT_DIR / 'direction_comparison.csv', index=False, encoding='utf-8-sig')

summary = pd.DataFrame([{
    'qwen_mean_chrf2': metrics.loc[metrics.model_name == 'qwen3_4b_q8', 'chrf2'].mean(),
    'best_m2m100_mean_chrf2': direction_comparison.best_m2m100_chrf2.mean(),
    'mean_delta_vs_best_m2m100': direction_comparison.qwen_minus_best_m2m100.mean(),
    'directions_qwen_wins': int((direction_comparison.qwen_minus_best_m2m100 > 0).sum()),
    'directions_qwen_loses': int((direction_comparison.qwen_minus_best_m2m100 < 0).sum()),
    'recommend_qwen_for_uzbek': bool((direction_comparison.qwen_minus_best_m2m100.mean() > 0) and ((direction_comparison.qwen_minus_best_m2m100 < -2).sum() <= 1)),
}])
summary.to_csv(RESULT_DIR / 'selection_summary.csv', index=False, encoding='utf-8-sig')
display(direction_comparison.round(4))
display(win_loss.round(4))
display(summary.round(4))


model_name,direction,m2m100_1_2b,m2m100_418m,qwen3_4b_q8,best_m2m100_chrf2,qwen_minus_best_m2m100
0,en-uz,13.9508,15.5223,14.9484,15.5223,-0.5740
1,ru-uz,11.6773,15.6118,23.1333,15.6118,7.5215
2,uz-en,20.7388,22.0329,43.3740,22.0329,21.3411
3,uz-ru,18.0590,18.8534,36.3225,18.8534,17.4691
4,uz-zh,4.7236,5.3828,16.2557,5.3828,10.8729
5,zh-uz,11.9627,15.0302,24.4121,15.0302,9.3818


,direction,comparison,mean_sentence_chrf2_delta,qwen_wins,ties,qwen_losses
0,en-uz,qwen3_4b_q8_minus_m2m100_418m,-0.6400,21,0,29
1,en-uz,qwen3_4b_q8_minus_m2m100_1_2b,1.0070,22,0,28
2,ru-uz,qwen3_4b_q8_minus_m2m100_418m,7.6996,38,2,10
3,ru-uz,qwen3_4b_q8_minus_m2m100_1_2b,11.5224,42,1,7
4,uz-en,qwen3_4b_q8_minus_m2m100_418m,20.9008,46,0,4
5,uz-en,qwen3_4b_q8_minus_m2m100_1_2b,21.9433,45,1,4
6,uz-ru,qwen3_4b_q8_minus_m2m100_418m,17.3763,48,0,2
7,uz-ru,qwen3_4b_q8_minus_m2m100_1_2b,18.0691,48,0,2
8,uz-zh,qwen3_4b_q8_minus_m2m100_418m,13.7664,44,0,6
9,uz-zh,qwen3_4b_q8_minus_m2m100_1_2b,13.7985,43,0,7


,qwen_mean_chrf2,best_m2m100_mean_chrf2,mean_delta_vs_best_m2m100,directions_qwen_wins,directions_qwen_loses,recommend_qwen_for_uzbek
0,26.4077,15.4056,11.0021,5,1,True


In [10]:
qwen_diag = all_predictions[all_predictions.model_name == 'qwen3_4b_q8'].copy()
qwen_diag['sentence_chrf2'] = [sacrebleu.sentence_chrf(str(p), [str(r)], word_order=2).score for p, r in zip(qwen_diag.prediction, qwen_diag.reference)]
problem_rows = qwen_diag.sort_values(['has_repetition', 'hit_max_tokens', 'abnormal_length', 'sentence_chrf2'], ascending=[False, False, False, True])
problem_rows.to_csv(RESULT_DIR / 'qwen_diagnostic_rows.csv', index=False, encoding='utf-8-sig')
display(problem_rows[['eval_id', 'src_lang', 'tgt_lang', 'source', 'reference', 'prediction', 'sentence_chrf2', 'has_repetition', 'abnormal_length']].head(24))

manifest = {
    'created_at': datetime.now().isoformat(timespec='seconds'),
    'model_repo': MODEL_REPO,
    'model_file': str(MODEL_FILE),
    'model_quantization': 'Q8_0',
    'license': 'Apache-2.0',
    'llama_server': str(LLAMA_SERVER),
    'llama_cuda_version': LLAMA_CUDA_VERSION,
    'benchmark_file': str(benchmark_path),
    'samples': len(benchmark_df),
    'directions': sorted(expected_directions),
    'generation': {'temperature': 0, 'top_p': 1, 'seed': 42, 'max_tokens': 256, 'context': 2048},
    'system_prompt': SYSTEM_PROMPT,
}
(RESULT_DIR / 'run_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')

if 'server_process' in globals() and server_process.poll() is None:
    server_process.terminate()
    try:
        server_process.wait(timeout=15)
    except subprocess.TimeoutExpired:
        server_process.kill()
        server_process.wait(timeout=10)
if 'server_log_handle' in globals() and not server_log_handle.closed:
    server_log_handle.close()
print('分析结果已保存，Notebook 启动的模型服务已关闭。')
print('结果目录:', RESULT_DIR)


,eval_id,src_lang,tgt_lang,source,reference,prediction,sentence_chrf2,has_repetition,abnormal_length
663,ru-uz-0397,ru,uz,"На земле Ханаана не было больших лесов, поэтом...","Kan'on yerlarida keng o'rmonlar yo'q edi, shu ...",Xanaan davriyida katta qo'shimcha qo'shimcha q...,4.957125,True,True
604,en-uz-0091,en,uz,The scientists were able to conclude that the ...,Olimlar qora materiya boshqa qora materiyaga x...,Aloqatliy tilga o'tkazish:\nAloqatliy tilga o'...,7.685738,True,True
896,zh-uz-0913,zh,uz,对于那些计划怎么度过空档年的人来说，游学成为越来越受欢迎的选择。,Akademik ta'tilni rejalashtirayotganlar orasid...,Ko'proq yoshlarda bo'sh yil uchun planlash ker...,9.213196,True,True
887,zh-uz-0771,zh,uz,除了一艘英国巡航舰外，其余船只悉数沉没。近 200 名美国人和德国人丧失生命。,Bitta Britaniya kreyseridan boshqa barcha kema...,Boshqa barcha qo'shimcha qurilma yoki qurilma ...,9.320682,True,True
861,zh-uz-0356,zh,uz,要把卫星或望远镜送入太空，需要一枚超过 100 英尺高的巨型火箭。,Sun'iy yo'ldosh yoki teleskopni kosmosga joyla...,Boshqaruv satelllisi yoki teleskopni uzunlikda...,10.914839,True,True
603,en-uz-0085,en,uz,Commons Administrator Adam Cuerden expressed h...,Parlamentning quyi palatasi ma'muri Adam Kuerd...,"Kommunalar administratori Adam Cuerden, aynan ...",11.655127,True,True
889,zh-uz-0814,zh,uz,如果你在宣布推迟之前预订了 2020 年的航班和住宿，则可能会面临棘手的情况。,Agar siz kechiktirish e'lon qilinishidan avval...,Agar 2020-yil bo'ylab bo'lib o'tgan kiritish v...,11.950273,True,True
697,ru-uz-0950,ru,uz,Севернее посетите также великий Храм Фатимской...,Yana shimolda butun dunyoga mashhur bo'lgan Ma...,"Ko'pincha, Katedrali (usupalinishi) - Mariya K...",12.329517,True,True
634,en-uz-0746,en,uz,The Chaco region was home to other groups of i...,Chako mintaqasida Guaykuru va Payagua kabi mah...,Chako regioni Guaycuru va Payaguá kabi boshqa ...,13.159960,True,True
899,zh-uz-0994,zh,uz,内陆水道可以作为假期游玩的一个不错的主题。,Ichki suv yo'llari ta'til uchun yaxshi mavzu b...,Ichki suv yo'llari yengi o'ylab ko'rgan bo'lga...,13.301970,True,True


分析结果已保存，Notebook 启动的模型服务已关闭。
结果目录: D:\dev\projects\fourlang_translation\results\qwen3_4b_q8_uzbek_selection
